In [1]:
from pathlib import Path
import pandas as pd
import re
from datetime import datetime

input_roots = [
    Path("raw_data_channel-#newdata"),
    Path("raw_data_states/2025-04-11"),
]

valid_suffixes = {".csv", ".xlsx", ".xls"}
input_files = []
for root in input_roots:
    attachments_dir = root / "attachments"
    scan_dir = attachments_dir if attachments_dir.exists() else root
    input_files.extend(
        [
            p
            for p in scan_dir.iterdir()
            if p.is_file() and not p.name.startswith(".") and p.suffix.lower() in valid_suffixes
        ]
    )
input_files = sorted(input_files)

frames = []
failed_files = []

for file_path in input_files:
    print(file_path)
    try:
        if file_path.suffix.lower() == ".csv":
            df = pd.read_csv(file_path, low_memory=False, dtype="string")
        else:
            # openpyxl handles modern Excel files (.xlsx)
            df = pd.read_excel(file_path, engine="openpyxl", dtype="string")

        # Track source file for traceability.
        df["source_file"] = str(file_path)
        frames.append(df)
    except Exception as exc:
        failed_files.append((str(file_path), str(exc)))

if not frames:
    raise RuntimeError("No files were successfully read.")

print(f"Merged {len(frames)} files")
merged_df = pd.concat(frames, ignore_index=True, sort=False)
available_columns = merged_df.columns
print(f"Rows: {len(merged_df):,} | Columns: {len(available_columns):,}")
# this ddupe here is required because the download consisted of multiple of the same file e.g.
# Detail2026032916241856.xlsx 
# slackdump was used to download so it attached a prefix F0###### on the file so it dosen't look like a dupe but it is indeed a dupe, uploaded more than once to the #newdata channel
merged_df = merged_df.sort_values(by=["source_file"]).drop_duplicates(subset=[col for col in available_columns if col != "source_file"])
print("No. of unique files:", len(merged_df["source_file"].unique()))
print("Rows after dropping duplicates:", len(merged_df))

raw_data_channel-#newdata/F0APCRU194P-Detail2026032919280956.xlsx
raw_data_channel-#newdata/F0APCTHH2P9-Detail2026032912453140.xlsx
raw_data_channel-#newdata/F0APCTKJSAK-Detail2026032912481489.xlsx
raw_data_channel-#newdata/F0APCTMQMJB-Detail2026032912553786.xlsx
raw_data_channel-#newdata/F0APCTSNJLX-Detail2026032913281910.xlsx
raw_data_channel-#newdata/F0APCU2DE87-Detail2026032913555413.xlsx
raw_data_channel-#newdata/F0APCU2LBJT-Detail2026032913514809.xlsx
raw_data_channel-#newdata/F0APCU4CGB1-Detail2026032914064873.xlsx
raw_data_channel-#newdata/F0APCUCFJB1-Detail2026032914370689.xlsx
raw_data_channel-#newdata/F0APCUFB15M-Detail2026032914444506.xlsx
raw_data_channel-#newdata/F0APCUG9BDM-Detail2026032914490487.xlsx
raw_data_channel-#newdata/F0APCULD44F-Detail2026032914533481.xlsx
raw_data_channel-#newdata/F0APCUNGT71-Detail2026032915483127.xlsx
raw_data_channel-#newdata/F0APCUUMUEB-Detail2026032916165801.xlsx
raw_data_channel-#newdata/F0APCV06R2T-Detail2026032916165801.xlsx
raw_data_c

In [2]:
# Map `merged_df` (channel export) to the final schema used in `10church_data.ipynb`.
# `master_df` is an alias for the same concatenated frame.

master_df = merged_df

TARGET_COLS = [
    "source_file",
    "record_id",
    "first_name",
    "last_name",
    "job_title",
    "email",
    "mobile_phone",
    "company_linkedin",
    "facebook",
    "twitter",
    "work_phone",
    "industry",
    "company_name",
    "company_website",
    "company_address",
    "company_zipcode",
    "company_employee_size_actual",
    "company_city",
    "company_state",
    "company_revenue",
    "company_location",
    "company_founded_at",
    "gender",
    "company_zipfour",
    "county",
    "company_description",
    "primary_sic_code",
    "primary_sic_code_description",
    "primary_naics",
    "primary_naics_description",
    "cuisine_code",
    "cuisine_code_description",
    "location_sales_volume_range",
    "location_sales_volume_actual",
    "company_employee_size_range",
    "company_sales_volume_range",
    "company_sales_volume_actual",
    "business_type",
    "credit_cards_accepted",
    "linkedin",
    "landline_phone",
    "home_address",
    "home_city",
    "home_state",
    "home_zipcode",
    "state_voter_id",
    "age",
    "age_range",
    "party_description",
    "ethnic_group",
    "us_congressional_district",
    "state_senate_district",
    "state_legislative_district",
    "state_house_district",
    "precinct",
    "county_commissioner_district",
    "county_supervisorial_district",
    "language_code",
    "marital_status",
    "religion_code",
    "presence_of_children_in_household",
    "household_net_worth",
    "veteran_in_household",
    "voting_performance_even_year_general",
    "voting_performance_even_year_primary",
    "voting_performance_even_year_general_and_primary",
    "voting_performance_minor_election",
    "primary_n_of_4",
    "general_2024",
    "primary_2024",
    "general_2022",
    "primary_2022",
    "general_2020",
    "primary_2020",
]


def map_channel_to_final_schema(
    df: pd.DataFrame,
) -> tuple[pd.DataFrame, frozenset[str], frozenset[str]]:
    """Project channel detail columns onto TARGET_COLS; unmapped targets are pd.NA."""
    df = df.copy()
    df.columns = [str(c).strip() for c in df.columns]

    used_source_cols = set()

    def src(col_name: str):
        used_source_cols.add(col_name)
        return df[col_name]

    linked_in = src("Linked-In")

    _column_map = {
        "source_file": src("source_file"),
        "record_id": pd.NA,
        "first_name": src("Executive First Name"),
        "last_name": src("Executive Last Name"),
        "job_title": src("Executive Title"),
        "email": pd.NA,
        "mobile_phone": pd.NA,
        "company_linkedin": linked_in,
        "facebook": src("Facebook"),
        "twitter": src("Twitter"),
        "work_phone": src("Phone Number Combined"),
        "industry": src("Primary NAICS Description"),
        "company_name": src("Company Name"),
        "company_website": src("Website"),
        "company_address": src("Address"),
        "company_zipcode": src("ZIP Code"),
        "company_employee_size_actual": src("Location Employee Size Actual"),
        "company_city": src("City"),
        "company_state": src("State"),
        "company_revenue": src("Corporate Sales Volume Actual"),
        "company_location": src("Metro Area"),
        "company_founded_at": src("Year Established"),
        "gender": src("Executive Gender"),
        "company_zipfour": src("ZIP Four"),
        "county": src("County"),
        "company_description": src("Company Description"),
        "primary_sic_code": src("Primary SIC Code"),
        "primary_sic_code_description": src("Primary SIC Description"),
        "primary_naics": src("Primary NAICS"),
        "primary_naics_description": src("Primary NAICS Description"),
        "cuisine_code": src("Cuisine Code"),
        "cuisine_code_description": src("Cuisine Code Description"),
        "location_sales_volume_range": src("Location Sales Volume Range"),
        "location_sales_volume_actual": src("Location Sales Volume Actual"),
        "company_employee_size_range": src("Location Employee Size Range"),
        "company_sales_volume_range": src("Corporate Sales Volume Range"),
        "company_sales_volume_actual": src("Corporate Sales Volume Actual"),
        "business_type": src("Type of Business"),
        "credit_cards_accepted": src("Credit Cards Accepted"),
        "linkedin": linked_in,
        "landline_phone": src("Toll Free Number Combined"),
        "home_address": src("Mailing Address"),
        "home_city": src("Mailing City"),
        "home_state": src("Mailing State"),
        "home_zipcode": src("Mailing Zip Code"),
        "state_voter_id": pd.NA,
        "age": pd.NA,
        "age_range": pd.NA,
        "party_description": pd.NA,
        "ethnic_group": pd.NA,
        "us_congressional_district": pd.NA,
        "state_senate_district": pd.NA,
        "state_legislative_district": pd.NA,
        "state_house_district": pd.NA,
        "precinct": pd.NA,
        "county_commissioner_district": pd.NA,
        "county_supervisorial_district": pd.NA,
        "language_code": pd.NA,
        "marital_status": pd.NA,
        "religion_code": pd.NA,
        "presence_of_children_in_household": pd.NA,
        "household_net_worth": pd.NA,
        "veteran_in_household": pd.NA,
        "voting_performance_even_year_general": pd.NA,
        "voting_performance_even_year_primary": pd.NA,
        "voting_performance_even_year_general_and_primary": pd.NA,
        "voting_performance_minor_election": pd.NA,
        "primary_n_of_4": pd.NA,
        "general_2024": pd.NA,
        "primary_2024": pd.NA,
        "general_2022": pd.NA,
        "primary_2022": pd.NA,
        "general_2020": pd.NA,
        "primary_2020": pd.NA,
    }
    assert set(_column_map) == set(TARGET_COLS)
    pieces = {k: _column_map[k] for k in TARGET_COLS}

    assert list(pieces.keys()) == TARGET_COLS

    out = pd.DataFrame(pieces, index=df.index)
    assert list(out.columns) == TARGET_COLS

    mapped_target_cols = frozenset(
        col_name for col_name, value in _column_map.items() if isinstance(value, pd.Series)
    )
    return out, frozenset(used_source_cols), mapped_target_cols


remapped_df, mapped_source_cols, mapped_target_cols = map_channel_to_final_schema(master_df)
remapped_df["record_id"] = range(2000000, 2000000 + len(remapped_df))
remapped_df = remapped_df.sort_values(by=["company_name", "first_name", "last_name"])

# --- validation summary ---
columns_with_no_data = [c for c in TARGET_COLS if remapped_df[c].isna().all()]
missing_in_remapped_df = [c for c in TARGET_COLS if c not in remapped_df.columns]

print(f"remapped_df shape: {remapped_df.shape}")
print(f"Columns (expected {len(TARGET_COLS)}): {len(remapped_df.columns)}")
print(f"Missing columns in remapped_df: {len(missing_in_remapped_df)}-> {missing_in_remapped_df}")
print(f"Columns with no data: {len(columns_with_no_data)} -> {columns_with_no_data}")

_source_column_names = [str(c).strip() for c in master_df.columns]
_unmapped_source_columns = sorted(c for c in _source_column_names if c not in mapped_source_cols)
print(
    f"Source columns not mapped to any target ({len(_unmapped_source_columns)}): {_unmapped_source_columns}"
)



remapped_df shape: (146050, 74)
Columns (expected 74): 74
Missing columns in remapped_df: 0-> []
Columns with no data: 31 -> ['email', 'mobile_phone', 'state_voter_id', 'age', 'age_range', 'party_description', 'ethnic_group', 'us_congressional_district', 'state_senate_district', 'state_legislative_district', 'state_house_district', 'precinct', 'county_commissioner_district', 'county_supervisorial_district', 'language_code', 'marital_status', 'religion_code', 'presence_of_children_in_household', 'household_net_worth', 'veteran_in_household', 'voting_performance_even_year_general', 'voting_performance_even_year_primary', 'voting_performance_even_year_general_and_primary', 'voting_performance_minor_election', 'primary_n_of_4', 'general_2024', 'primary_2024', 'general_2022', 'primary_2022', 'general_2020', 'primary_2020']
Source columns not mapped to any target (342): ['Accounting Expenses', 'Advertising  Expenses', 'Affiliated Locations', 'Affiliated Records', 'Carrier Route', 'Census Blo

In [3]:
# cleaning data - 30s
clean_df = remapped_df.copy()

clean_df = clean_df[clean_df["first_name"].notna() & clean_df["last_name"].notna()]

clean_df["company_zipcode"] = "'" + clean_df["company_zipcode"].str.rjust(5, "0")

strip_cols = ["company_address", "home_address"]
for col in strip_cols:
    clean_df[col] = clean_df[col].str.strip()

# make the column values lowercase
lowercase_cols = ["email", "linkedin", "company_linkedin", "facebook", "twitter"]
for col in lowercase_cols:
    clean_df[col] = clean_df[col].str.lower()

# make the gender column uppercase
uppercase_cols = ["gender"]
for col in uppercase_cols:
    clean_df[col] = clean_df[col].str.upper()

clean_df["gender"] = clean_df["gender"].replace({"MALE": "M", "FEMALE": "F"})

# columns to title case
title_cols = ["first_name", "last_name", "job_title", "industry"]
for col in title_cols:
    clean_df[col] = clean_df[col].str.title()

# remove all phone numbers that contain e+ or are less than 6z digits
for col in ["mobile_phone", "work_phone", "landline_phone"]:
    mask = clean_df[col].astype(str).str.lower().str.contains("e+", regex=False, na=False)
    mask = mask | (clean_df[col].astype(str).str.len() <= 6) | (clean_df[col] == "Not Available")
    clean_df.loc[mask, col] = None

clean_df["company_employee_size_actual"] = clean_df["company_employee_size_actual"].astype("Int64")

# remove all email values where it does not contain @
clean_df.loc[~clean_df["email"].astype(str).str.contains("@", na=False), "email"] = None

url_pattern = r'(?:https?:/|www\.)'
cols_to_check_for_links = [x for x in clean_df.columns.tolist() if x not in ['source_file', 'record_id', 'company_linkedin', 'facebook', 'twitter', 'company_website', 'linkedin']]
for col in cols_to_check_for_links:
    contains_url = clean_df[col].astype("string").str.contains(url_pattern, case=False, na=False, regex=True)
    clean_df.loc[contains_url, col] = None

# Helper functions for extracting valid social links - thsi can take up to 3 minutes to run
def extract_linkedin(val):
    if pd.isna(val):
        return None
    s = str(val).strip().lower()
    if "linkedin.com/company" in s:
        return None  # Exclude company linkedin from personal
    if "linkedin" in s:
        return s
    return None

def extract_company_linkedin(val):
    if pd.isna(val):
        return None
    s = str(val).strip().lower()
    if "linkedin.com/company" in s:
        return s
    return None

def extract_facebook(val):
    if pd.isna(val):
        return None
    s = str(val).strip().lower()
    if "facebook" in s:
        return s
    return None

def extract_twitter(val):
    if pd.isna(val):
        return None
    s = str(val).strip().lower()
    if "twitter" in s:
        return s
    return None

def is_valid_linkedin(val):
    return extract_linkedin(val) is not None

def is_valid_facebook(val):
    return extract_facebook(val) is not None

def is_valid_twitter(val):
    return extract_twitter(val) is not None

# For each row, find all valid values and move to correct columns.
# If the url is not linkedin/company_linkedin/facebook/twitter, put it under website
def reorganize_socials(row):
    sources = {
        "linkedin": row["linkedin"],
        "company_linkedin": row["company_linkedin"],
        "facebook": row["facebook"],
        "twitter": row["twitter"],
        "company_website": row["company_website"]
    }
    vals = list(sources.values())

    linkedin_vals = [extract_linkedin(v) for v in vals if extract_linkedin(v) is not None]
    company_linkedin_vals = [extract_company_linkedin(v) for v in vals if extract_company_linkedin(v) is not None]
    facebook_vals = [extract_facebook(v) for v in vals if extract_facebook(v) is not None]
    twitter_vals = [extract_twitter(v) for v in vals if extract_twitter(v) is not None]

    # Track which URLs are already assigned to socials
    assigned_urls = set()
    if linkedin_vals:
        assigned_urls.add(linkedin_vals[0])
    if company_linkedin_vals:
        assigned_urls.add(company_linkedin_vals[0])
    if facebook_vals:
        assigned_urls.add(facebook_vals[0])
    if twitter_vals:
        assigned_urls.add(twitter_vals[0])

    # Assign the first valid for each, or None
    row["linkedin"] = linkedin_vals[0] if linkedin_vals else None
    row["company_linkedin"] = company_linkedin_vals[0] if company_linkedin_vals else None
    row["facebook"] = facebook_vals[0] if facebook_vals else None
    row["twitter"] = twitter_vals[0] if twitter_vals else None

    # Now handle website: any http(s) url not used for above social fields
    def is_url(val):
        if pd.isna(val):
            return False
        s = str(val).strip().lower()
        return s.startswith("http") or "www" in s or ".co" in s
    website_candidates = []
    for v in vals:
        if is_url(v):
            s = str(v).strip().lower()
            # Not any valid social
            if (
                (extract_linkedin(s) is None)
                and (extract_company_linkedin(s) is None)
                and (extract_facebook(s) is None)
                and (extract_twitter(s) is None)
            ):
                website_candidates.append(s)
    # Set to website if one found
    row["company_website"] = website_candidates[0] if website_candidates else None

    return row

clean_df = clean_df.apply(reorganize_socials, axis=1)

# Fix misspellings in job titles, can take up to 30s to run
job_title_filters = {
    "Owner": [
        'owner',
        'onwer',
        'ownwer',
        'owneer',
        'ownder',
        'ownrer',
        'owmer',
        'ownew',
        'onwer',
        'owener',
        'ownr',
        'ower',
    ],
    "President": [
        'president',
        'presiden',
        'presdient',
        'presidnet',
        'presidant',
        'pressident',
        'presient',
        'presudent',
        'presidnt',
        'presedent',
        'pressdent',
        'preisdent',
        'pesident',
        'presid',
        'pres',
        'prez',
    ],
    "Chief Executive Officer": [
        'ceo',
        'chiefexecutiveofficer',
        'chiefexectutiveofficer',
        'chiefexecutingofficer',
        'chiefexectiveofficer',
        'chiefexcutiveofficer',
        'chiefexectuiveofficer',
        'chiefexecutionofficer',
        'chiefofexecutiveofficer',
        'chiefexecutiveoffice',
        'chiefexecutive',
        'cheifexecutive',
    ],
    "Founder": [
        'founder',
        'foundr',
        'foudner',
        'foudnerr',
        'foudner',
        'foudner',
        'foudner',
        'funder',
    ]
}

# Clean up job titles by replacing misspellings with canonical versions, can take up to 50s to run
for canonical, variations in job_title_filters.items():
    for variation in variations:
        # Use word boundaries to avoid partial matches, case-insensitive
        pattern = r'\b' + re.escape(variation) + r'\b'
        mask = clean_df["job_title"].str.contains(pattern, case=False, regex=True, na=False)
        if mask.any():
            clean_df.loc[mask, "job_title"] = clean_df.loc[mask, "job_title"].str.replace(pattern, canonical, case=False, regex=True)
            break  # Move onto the next canonical once a variation is found



In [4]:
test_mobile_alpha = clean_df[(clean_df["mobile_phone"].fillna("").str.contains(r"[A-Za-z]")) & (clean_df["mobile_phone"].notna())]
if len(test_mobile_alpha) > 0:
    print("TEST FAILED: MOBILE ALPHA")
else:
    print("TEST PASSED: MOBILE ALPHA")

test_landline_alpha = clean_df[(clean_df["landline_phone"].fillna("").str.contains(r"[A-Za-z]")) & (clean_df["landline_phone"].notna())]
if len(test_landline_alpha) > 0:
    print("TEST FAILED: LANDLINE ALPHA")
else:
    print("TEST PASSED: LANDLINE ALPHA")

test_work_alpha = clean_df[(clean_df["work_phone"].fillna("").str.contains(r"[A-Za-z]")) & (clean_df["work_phone"].notna())]
if len(test_work_alpha) > 0:
    print("TEST FAILED: WORK ALPHA")
else:
    print("TEST PASSED: WORK ALPHA")

test_email = clean_df[(~clean_df["email"].fillna("").str.contains("@")) & (clean_df["email"].notna())]
if len(test_email) > 0:
    print("TEST FAILED: EMAIL")
else:
    print("TEST PASSED: EMAIL")

test_linkedin = clean_df[(~clean_df["linkedin"].fillna("").str.contains("linkedin")) & (clean_df["linkedin"].notna())]
if len(test_linkedin) > 0:
    print("TEST FAILED: LINKEDIN")
else:
    print("TEST PASSED: LINKEDIN")
test_company_linkedin = clean_df[(~clean_df["company_linkedin"].fillna("").str.contains("linkedin")) & (clean_df["company_linkedin"].notna())]
if len(test_company_linkedin) > 0:
    print("TEST FAILED: COMPANY LINKEDIN")
else:
    print("TEST PASSED: COMPANY LINKEDIN")

test_facebook = clean_df[(~clean_df["facebook"].fillna("").str.contains("facebook")) & (clean_df["facebook"].notna())]
if len(test_facebook) > 0:
    print("TEST FAILED: FACEBOOK")
else:
    print("TEST PASSED: FACEBOOK")

test_twitter = clean_df[(~clean_df["twitter"].fillna("").str.contains("twitter")) & (clean_df["twitter"].notna())]
if len(test_twitter) > 0:
    print("TEST FAILED: TWITTER")
else:
    print("TEST PASSED: TWITTER")

test_gender = clean_df[(~clean_df["gender"].isin(["M", "F"])) & (clean_df["gender"].notna())]
if len(test_gender) > 0:
    print("TEST FAILED: GENDER")
else:
    print("TEST PASSED: GENDER")

test_email = clean_df[(~clean_df["email"].fillna("").str.contains("@")) & (clean_df["email"].notna())]
if len(test_email) > 0:
    print(f"TEST FAILED: EMAIL {len(test_email)}")
else:
    print("TEST PASSED: EMAIL")

TEST PASSED: MOBILE ALPHA
TEST PASSED: LANDLINE ALPHA
TEST PASSED: WORK ALPHA
TEST PASSED: EMAIL
TEST PASSED: LINKEDIN
TEST PASSED: COMPANY LINKEDIN
TEST PASSED: FACEBOOK
TEST PASSED: TWITTER
TEST PASSED: GENDER
TEST PASSED: EMAIL


In [5]:
datetime_now = datetime.now().strftime("%Y-%m-%d")

# Apply the same split logic used in 5split_job_title.ipynb.
owner_president_ceo_founder_mask = clean_df["job_title"].str.contains(
    "founder|president|chief executive officer|owner",
    na=False,
    case=False,
)
no_assistant_vice_mask = ~clean_df["job_title"].str.contains("assistant|vice", case=False, na=False)
filtered_data_mask = owner_president_ceo_founder_mask & no_assistant_vice_mask

filtered_data = clean_df[filtered_data_mask]
other_data = clean_df[~filtered_data_mask]

print("Filtered Data:", len(filtered_data))
print("Other Data:", len(other_data))


Filtered Data: 95656
Other Data: 38300


In [ ]:
# filter out schools
filtered_data = filtered_data[~filtered_data["company_name"].str.lower().str.contains("school|elementary|university|college")]
print("Filtered Data:", len(filtered_data))
print("Other Data:", len(other_data))

94927


In [17]:
import splink.comparison_library as cl
from splink import Linker, SettingsCreator, block_on, DuckDBAPI

# Start from this notebook's filtered output.
df_for_dedupe = filtered_data.copy().astype("string")

# Clean key columns for matching.
df_for_dedupe["first_name_clean"] = (
    df_for_dedupe["first_name"].fillna("").str.replace(r"[^a-zA-Z]", "", regex=True).str.strip().str.lower()
)
df_for_dedupe["last_name_clean"] = (
    df_for_dedupe["last_name"].fillna("").str.replace(r"[^a-zA-Z]", "", regex=True).str.strip().str.lower()
)
df_for_dedupe["forename_surname_concat_col_name"] = (
    df_for_dedupe["first_name_clean"] + " " + df_for_dedupe["last_name_clean"]
)
df_for_dedupe["company_name_clean"] = (
    df_for_dedupe["company_name"].fillna("").str.replace(r"[^a-zA-Z0-9]", "", regex=True).str.strip().str.lower()
)

print(f"Rows entering Splink dedupe: {len(df_for_dedupe):,}")

Rows entering Splink dedupe: 94,927


In [18]:
# Define Splink settings.
settings = SettingsCreator(
    link_type="dedupe_only",
    unique_id_column_name="record_id",
    blocking_rules_to_generate_predictions=[
        block_on("first_name_clean", "last_name_clean"),
        block_on("first_name_clean", "company_name_clean"),
    ],
    comparisons=[
        cl.NameComparison("first_name_clean"),
        cl.NameComparison("last_name_clean"),
        cl.NameComparison("company_name_clean"),
        cl.ExactMatch("company_zipcode"),
    ],
    retain_intermediate_calculation_columns=True,
)

# Initialize linker with DuckDB backend.
linker = Linker(df_for_dedupe, settings, db_api=DuckDBAPI())
print("Splink linker initialized")

Splink linker initialized


In [19]:
# Train m/u parameters.
linker.training.estimate_probability_two_random_records_match(
    [block_on("first_name_clean", "last_name_clean")],
    recall=0.7,
)

linker.training.estimate_u_using_random_sampling(max_pairs=int(1e8))

linker.training.estimate_parameters_using_expectation_maximisation(
    block_on("first_name_clean", "last_name_clean")
)
linker.training.estimate_parameters_using_expectation_maximisation(
    block_on("first_name_clean", "company_name_clean")
)


Probability two random records match is estimated to be  4.86e-06.
This means that amongst all possible pairwise record comparisons, one in 205,852.37 are expected to match.  With 4,505,520,201 total possible comparisons, we expect a total of around 21,887.14 matching pairs
----- Estimating u probabilities using random sampling -----

Estimated u probabilities using random sampling

Your model is not yet fully trained. Missing estimates for:
    - first_name_clean (no m values are trained).
    - last_name_clean (no m values are trained).
    - company_name_clean (no m values are trained).
    - company_zipcode (no m values are trained).

----- Starting EM training session -----

Estimating the m probabilities of the model by blocking on:
(l."first_name_clean" = r."first_name_clean") AND (l."last_name_clean" = r."last_name_clean")

Parameter estimates will be made for the following comparison(s):
    - company_name_clean
    - company_zipcode

Parameter estimates cannot be made for the

<EMTrainingSession, blocking on (l."first_name_clean" = r."first_name_clean") AND (l."company_name_clean" = r."company_name_clean"), deactivating comparisons first_name_clean, company_name_clean>

In [20]:

linker.visualisations.match_weights_chart()


/Users/rayleigh/Projects/us-political-data-merge-robv/.venv/lib/python3.13/site-packages/altair/vegalite/v6/api.py:4124: UserWarning: Automatically deduplicated selection parameter with identical configuration. If you want independent parameters, explicitly name them differently (e.g., name='param1', name='param2'). See https://github.com/vega/altair/issues/3891
  return _tp.from_dict(dct, validate=validate)


alt.VConcatChart(...)

In [34]:
threshold = 0.98

# Predict likely matching pairs.
print("Running predictions...")
predictions = linker.inference.predict(threshold_match_probability=threshold)
pairwise_predictions = predictions.as_pandas_dataframe()
print("Predictions complete")

# Cluster pairwise predictions into entities.
clusters = linker.clustering.cluster_pairwise_predictions_at_threshold(
    predictions,
    threshold_match_probability=threshold,
)

df_with_clusters = clusters.as_pandas_dataframe().astype("string")
print(f"Number of clusters: {df_with_clusters['cluster_id'].nunique():,}")
print(f"Records in clusters: {len(df_with_clusters):,}")

df_with_clusters = df_with_clusters.sort_values(by="cluster_id")
cluster_size = df_with_clusters.groupby("cluster_id")["cluster_id"].transform("size")
df_with_clusters.insert(0, "cluster_size", cluster_size.astype("Int64"))
df_with_clusters


Blocking time: 0.02 seconds
Predict time: 0.05 seconds

 -- WARNING --
You have called predict(), but there are some parameter estimates which have neither been estimated or specified in your settings dictionary.  To produce predictions the following untrained trained parameters will use default values.
Comparison: 'first_name_clean':
    m values not fully trained
Completed iteration 1, num edges remaining to process: 226
Completed iteration 2, num edges remaining to process: 10
Completed iteration 3, num edges remaining to process: 2
Completed iteration 4, num edges remaining to process: 0


Running predictions...
Predictions complete
Number of clusters: 90,714
Records in clusters: 94,927


,cluster_size,cluster_id,source_file,record_id,first_name,last_name,job_title,email,mobile_phone,company_linkedin,...,general_2024,primary_2024,general_2022,primary_2022,general_2020,primary_2020,first_name_clean,last_name_clean,forename_surname_concat_col_name,company_name_clean
59036,1,2000002,raw_data_channel-#newdata/F0APCRU194P-Detail20...,2000002,Walter,Springs,Owner,<NA>,<NA>,https://www.linkedin.com/company/walter-m.-spr...,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,walter,springs,walter springs,springsconstructioninc
59035,1,2000003,raw_data_channel-#newdata/F0APCRU194P-Detail20...,2000003,Jim,Adkins,President,<NA>,<NA>,<NA>,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,jim,adkins,jim adkins,springsconstructioninc
59063,1,2000004,raw_data_channel-#newdata/F0APCRU194P-Detail20...,2000004,Vinh,Vi,Owner,<NA>,<NA>,<NA>,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,vinh,vi,vinh vi,sproutcafe
59067,1,2000005,raw_data_channel-#newdata/F0APCRU194P-Detail20...,2000005,Kevin,Lee,Owner,<NA>,<NA>,<NA>,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,kevin,lee,kevin lee,sprucecafe
59071,1,2000007,raw_data_channel-#newdata/F0APCRU194P-Detail20...,2000007,Andi,Torres,Owner,<NA>,<NA>,<NA>,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,andi,torres,andi torres,sprucesalon
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
51494,1,2146045,raw_data_states/2025-04-11/washington.xlsx,2146045,Dave,Winters,Owner,<NA>,<NA>,<NA>,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,dave,winters,dave winters,swedishautomotiveinc
51495,1,2146046,raw_data_states/2025-04-11/washington.xlsx,2146046,Sarah,Zabel,Chief Executive Officer,<NA>,<NA>,http://www.linkedin.com/company/providence-hea...,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,sarah,zabel,sarah zabel,swedishbirthctredmonds
51517,1,2146047,raw_data_states/2025-04-11/washington.xlsx,2146047,Adrienne,Jeffery,Owner,<NA>,<NA>,<NA>,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,adrienne,jeffery,adrienne jeffery,sweetironwaffle
51265,1,2146048,raw_data_states/2025-04-11/washington.xlsx,2146048,Jordan,Voloshin,Chief Executive Officer,<NA>,<NA>,<NA>,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,jordan,voloshin,jordan voloshin,surlatableinc


In [35]:
def agg_join(series):
    return series.tolist()


def agg_longest(series):
    non_null = series.dropna()
    if len(non_null) == 0:
        return None
    return max(non_null, key=len)


def agg_phone_numbers(series):
    non_null = series.dropna()
    if len(non_null) == 0:
        return None

    mask = non_null.str.contains(r"[()]", regex=True, na=False)
    if mask.any():
        return non_null[mask].iloc[0]

    mask = non_null.str.contains(r"\+", regex=True, na=False)
    if mask.any():
        return non_null[mask].iloc[0]

    return max(non_null, key=len)


aggregation_rules = [
    (lambda col: col == "source_file", agg_join),
    (lambda col: col == "record_id", agg_join),
    (lambda col: "work_phone" in col.lower(), agg_phone_numbers),
    (lambda col: "mobile_phone" in col.lower(), agg_phone_numbers),
    (lambda col: "landline_phone" in col.lower(), agg_phone_numbers),
    (lambda col: "description" in col.lower(), agg_longest),
    (lambda col: "actual" in col.lower(), "max"),
    (lambda col: "volume" in col.lower(), "max"),
    (lambda col: "revenue" in col.lower(), "max"),
    (lambda col: col == "age", "max"),
]

default_agg = "first"

def get_agg_function(col_name):
    for condition, agg_func in aggregation_rules:
        if condition(col_name):
            return agg_func
    return default_agg


agg_cols = [col for col in df_with_clusters.columns if col != "cluster_id"]
agg_dict = {col: get_agg_function(col) for col in agg_cols}

print(f"Original rows: {len(df_with_clusters):,}")
df_deduped = df_with_clusters.groupby("cluster_id", dropna=False).agg(agg_dict).reset_index()
print(f"Deduplicated rows: {len(df_deduped):,}")
print(f"Duplicates removed: {len(df_with_clusters) - len(df_deduped):,}")

df_deduped.head()

Original rows: 94,927
Deduplicated rows: 90,714
Duplicates removed: 4,213


,cluster_id,cluster_size,source_file,record_id,first_name,last_name,job_title,email,mobile_phone,company_linkedin,...,general_2024,primary_2024,general_2022,primary_2022,general_2020,primary_2020,first_name_clean,last_name_clean,forename_surname_concat_col_name,company_name_clean
0,2000002,1,[raw_data_channel-#newdata/F0APCRU194P-Detail2...,[2000002],Walter,Springs,Owner,<NA>,<NA>,https://www.linkedin.com/company/walter-m.-spr...,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,walter,springs,walter springs,springsconstructioninc
1,2000003,1,[raw_data_channel-#newdata/F0APCRU194P-Detail2...,[2000003],Jim,Adkins,President,<NA>,<NA>,<NA>,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,jim,adkins,jim adkins,springsconstructioninc
2,2000004,1,[raw_data_channel-#newdata/F0APCRU194P-Detail2...,[2000004],Vinh,Vi,Owner,<NA>,<NA>,<NA>,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,vinh,vi,vinh vi,sproutcafe
3,2000005,1,[raw_data_channel-#newdata/F0APCRU194P-Detail2...,[2000005],Kevin,Lee,Owner,<NA>,<NA>,<NA>,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,kevin,lee,kevin lee,sprucecafe
4,2000007,1,[raw_data_channel-#newdata/F0APCRU194P-Detail2...,[2000007],Andi,Torres,Owner,<NA>,<NA>,<NA>,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,andi,torres,andi torres,sprucesalon


In [36]:
dedupe_output_path = f"csv_outputs/newdata_az_co_wa_filtered_deduped_{datetime_now}.csv"
clusters_output_path = f"csv_outputs/newdata_az_co_wa_filtered_clusters_{datetime_now}.csv"

pairwise_output_path = f"csv_outputs/newdata_az_co_wa_filtered_pairwise_predictions_{datetime_now}.csv"

pairwise_predictions.to_csv(pairwise_output_path, index=False)
df_with_clusters.to_csv(clusters_output_path, index=False)
df_deduped.to_csv(dedupe_output_path, index=False)

print("Saved pairwise predictions:", pairwise_output_path)
print("Saved clusters:", clusters_output_path)
print("Saved deduped output:", dedupe_output_path)

Saved pairwise predictions: csv_outputs/newdata_az_co_wa_filtered_pairwise_predictions_2026-04-14.csv
Saved clusters: csv_outputs/newdata_az_co_wa_filtered_clusters_2026-04-14.csv
Saved deduped output: csv_outputs/newdata_az_co_wa_filtered_deduped_2026-04-14.csv
